# Notebook 3 — Diagnostics on the saved v2 models (no new main experiments)

**Purpose.** Before building any new fusion method, measure what the existing 12 trained models actually do.
Everything here reuses the v2 checkpoints. The v2 folder is **only read**; all outputs go to `mosei_v3_diagnostics/`.

| Section | Question it answers | New training? |
|---|---|---|
| 5. Integrity check | Do the reloaded checkpoints reproduce the saved v2 predictions exactly? | No |
| 6. Modality-subset evaluation | How well does each trained model do when only text / audio / video / pairs are available? | No |
| 7. Leave-one-out modality utility | For each test clip, does adding audio (or video, or text) raise or lower the probability of the correct label? | No |
| 8. Gate vs. utility | Do the gate weights follow that measured usefulness, or only availability? | No |
| 9. Audio+video headroom | On how many clips does dropping audio+video fix or break the prediction, and what is the upper bound? | No |
| 10. Repeatability diagnostic | Does the number of CPU threads change the trained weights (a candidate cause of the v1/v2 difference)? | Yes, 9 short re-training runs (≈15–25 min), optional |

**Important definitions**
- *Subset evaluation* hides modalities **at test time** in a model trained on all three. It is **not** the same as a trained
  audio-only or video-only baseline; those still have to be trained separately.
- *Leave-one-out utility* of modality m for one clip: Δ_m = log p(correct label | all modalities) − log p(correct label | all except m).
  Δ_m > 0 means the modality helped on that clip. It uses the true label, so it is a **diagnostic**, not something usable at prediction time.
- For M2 (the RMSR adaptation), "removing" a modality means zero input, but its gate still gives that modality weight
  (Section 10 of v2 showed this). So M2's Δ values measure the effect of zero-filling, not of a clean removal.

**Runtime:** CPU + High-RAM. Sections 5–9 take a few minutes. Section 10 is controlled by `RUN_REPEATABILITY`.

In [3]:
import os, sys, time, json, random, pickle, platform, warnings, hashlib, itertools
warnings.filterwarnings("ignore", category=UserWarning)
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from scipy import stats
DEVICE = "cpu"
torch.backends.cudnn.deterministic = True
ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "torch_threads": torch.get_num_threads(), "os": platform.platform(),
       "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
try:
    ENV["cpu"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except Exception:
    ENV["cpu"] = platform.processor() or "unknown"
print(json.dumps(ENV, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "python": "3.13.18",
  "torch": "2.14.1+cpu",
  "numpy": "2.4.6",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "torch_threads": 4,
  "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "started_utc": "2026-11-24 07:51:41",
  "cpu": "Intel(R) Xeon(R) CPU @ 2.20GHz"
}


## 1. Paths and settings
The training settings are **read from the v2 run's own `run_info.json`**, so they cannot drift from the run being analysed.

In [4]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
V2_DIR = f"{BASE_OUT}/mosei_v2_cpu"                      # read only
V1_RESULTS = f"{BASE_OUT}/mosei_results_all_seeds.csv"     # read only
OUT_DIR = f"{BASE_OUT}/mosei_v3_diagnostics"
RUN_REPEATABILITY = True          # Section 10: set False to skip the short re-training runs
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]; V2_DIR = f"{BASE_OUT}/mosei_v2_cpu"
    V1_RESULTS = os.environ.get("EMOFAR_PREV", "none.csv"); OUT_DIR = f"{BASE_OUT}/mosei_v3_diagnostics"

assert os.path.isdir(V2_DIR), f"v2 folder not found: {V2_DIR}"
assert os.path.abspath(OUT_DIR) != os.path.abspath(V2_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
v2_info = json.load(open(f"{V2_DIR}/run_info.json"))
SEEDS = v2_info["config"]["SEEDS"]
# The authoritative training settings are the ones stored inside each checkpoint's log (written when it was trained).
logs = [json.load(open(f"{V2_DIR}/checkpoints/{m}_seed{s}_log.json")) for m in ["M1", "M2", "M3", "M4"] for s in SEEDS]
cfg = logs[0]["train_config"]
assert all(l["train_config"] == cfg for l in logs), "checkpoints were trained with different settings"
DATA_PATH = cfg["DATA_PATH"] if not TEST_MODE else os.environ["EMOFAR_MOSI"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
print("Training settings stored in the v2 checkpoints:", cfg, "| seeds:", SEEDS)
print("v2 run environment:", v2_info["environment"])
RUN_DIR = OUT_DIR                 # the unchanged v2 data cell below saves labels/ids into RUN_DIR -> the new folder
CKPT_DIR = f"{V2_DIR}/checkpoints"
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]
json.dump({"environment": ENV, "v2_train_config": cfg, "seeds": SEEDS}, open(f"{OUT_DIR}/run_info.json", "w"), indent=2)

Training settings stored in the v2 checkpoints: {'D_MODEL': 128, 'EPOCHS': 30, 'PATIENCE': 6, 'BATCH_SIZE': 64, 'LR': 0.0005, 'MOD_DROPOUT': 0.3, 'DATA_PATH': '/content/drive/MyDrive/datasets/CMU-MOSEI/aligned_50.pkl', 'n_train': 16326} | seeds: [42, 43, 44]
v2 run environment: {'python': '3.13.18', 'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'torch_threads': 4, 'os': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'started_utc': '2026-11-24 06:35:51'}


## 2. Load the features (code copied unchanged from v2)

In [5]:
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")

def prep(split):
    d = raw[split]
    mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
    feats = {"text": d["text"].astype(np.float32),
             "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
             "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
    score = d["regression_labels"].astype(np.float32)
    y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
    return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

data = {s: prep(s) for s in ["train", "valid", "test"]}
del raw

for m in ["audio", "vision"]:                                           # standardise with train statistics
    tr = data["train"]["feats"][m][data["train"]["mask"]]
    mu, sd = tr.mean(0), tr.std(0) + 1e-6
    for s in data:
        x = (data[s]["feats"][m] - mu) / sd
        data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
print("Feature sizes:", DIMS)
for s in data:
    counts = np.bincount(data[s]["y"], minlength=3)
    print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
# the labels must match the ones saved by v2
assert (np.load(f"{V2_DIR}/test_labels.npy") == data["test"]["y"]).all(), "test labels differ from the v2 run"
assert len(data["train"]["y"]) == cfg["n_train"], "training set size differs from the v2 run"

Loaded in 50s
Loaded in 6s
Feature sizes: {'text': 768, 'audio': 74, 'vision': 35}
train: 16,326 clips | negative 4,738 / neutral 3,540 / positive 8,048
valid:  1,871 clips | negative 506 / neutral 433 / positive 932
test :  4,659 clips | negative 1,350 / neutral 1,025 / positive 2,284


## 3. Model definitions (copied unchanged from v2)

In [6]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

## 4. Training/evaluation functions (copied unchanged from v2; `train` is used only in Section 10)

In [7]:
from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 5. Integrity check: reload the 12 checkpoints and compare with the saved v2 test predictions

In [8]:
trained = {}
rows = []
for name in MODELS:
    for seed in SEEDS:
        tag = f"{SHORT[name]}_seed{seed}"
        model = MODELS[name][0]().to(DEVICE)
        model.load_state_dict(torch.load(f"{CKPT_DIR}/{tag}.pt", map_location=DEVICE)); model.eval()
        trained[(name, seed)] = model
        saved = np.load(f"{CKPT_DIR}/{tag}_test_probs.npy")
        now = predict(model, data["test"])
        rows.append({"run": tag, "max_abs_prob_diff": float(np.abs(saved - now).max()),
                     "same_predictions": bool((saved.argmax(1) == now.argmax(1)).all())})
integrity = pd.DataFrame(rows); integrity.to_csv(f"{OUT_DIR}/integrity_check.csv", index=False)
print(integrity.to_string(index=False))
print("ALL CHECKPOINTS REPRODUCE THE SAVED PREDICTIONS" if integrity.same_predictions.all()
      else "MISMATCH — stop and investigate before using the diagnostics")

      run  max_abs_prob_diff  same_predictions
M1_seed42                0.0              True
M1_seed43                0.0              True
M1_seed44                0.0              True
M2_seed42                0.0              True
M2_seed43                0.0              True
M2_seed44                0.0              True
M3_seed42                0.0              True
M3_seed43                0.0              True
M3_seed44                0.0              True
M4_seed42                0.0              True
M4_seed43                0.0              True
M4_seed44                0.0              True
ALL CHECKPOINTS REPRODUCE THE SAVED PREDICTIONS


## 6. Modality-subset evaluation (test time only)
Each trained model is evaluated with every non-empty combination of available modalities, for all test clips.
Reminder: M3 was never trained with missing modalities; M4 was (modality dropout); M2 has no availability mask.

In [9]:
N_TEST = len(data["test"]["y"])
SUBSETS = [s for r in (1, 2, 3) for s in itertools.combinations(range(3), r)]
SUBSET_NAME = {s: "+".join(MODS[i][0].upper() for i in s) for s in SUBSETS}     # e.g. "T+A"

@torch.no_grad()
def probs_with(model, avail_row, return_gate=False):
    model.eval(); out, gates = [], []
    for i in range(0, N_TEST, 256):
        idx = np.arange(i, min(i + 256, N_TEST))
        avail = np.tile(np.asarray(avail_row, dtype=np.float32), (len(idx), 1))
        x, mask, av = to_batch(data["test"], idx, avail)
        logits, g = model(x, mask, av, return_gate=True)
        out.append(logits.softmax(-1).numpy())
        if g is not None: gates.append(g.numpy())
    return np.concatenate(out), (np.concatenate(gates) if gates else None)

subset_probs, gates_full = {}, {}
rows = []
for (name, seed), model in trained.items():
    for s in SUBSETS:
        if name.startswith("M1") and 0 not in s:
            continue                                              # text-only model has no input without text
        avail_row = [1.0 if i in s else 0.0 for i in range(3)]
        p, g = probs_with(model, avail_row)
        subset_probs[(name, seed, s)] = p
        if s == (0, 1, 2): gates_full[(name, seed)] = g
        r = evaluate(p, data["test"]); r.update(model=SHORT[name], seed=seed, available=SUBSET_NAME[s]); rows.append(r)
sub = pd.DataFrame(rows); sub.to_csv(f"{OUT_DIR}/subset_evaluation.csv", index=False)
sub_table = sub.groupby(["available", "model"]).macro_f1.agg(["mean", "std"]).unstack("model")
sub_table = sub_table.reindex([SUBSET_NAME[s] for s in SUBSETS])
print("Test macro-F1 (mean and SD over 3 training seeds) when only these modalities are available:")
sub_table.round(4)

Test macro-F1 (mean and SD over 3 training seeds) when only these modalities are available:


mean                             std                        
model          M1      M2      M3      M4      M1      M2      M3      M4
available                                                                
T          0.6319  0.6165  0.6215  0.6267  0.0037  0.0024  0.0071  0.0038
A             NaN  0.3162  0.3798  0.4002     NaN  0.0507  0.0047  0.0072
V             NaN  0.3408  0.3642  0.3945     NaN  0.0336  0.0195  0.0117
T+A        0.6319  0.6178  0.6185  0.6281  0.0037  0.0037  0.0092  0.0034
T+V        0.6319  0.6235  0.6286  0.6357  0.0037  0.0041  0.0105  0.0059
A+V           NaN  0.3605  0.3895  0.4176     NaN  0.0418  0.0102  0.0042
T+A+V      0.6319  0.6227  0.6287  0.6345  0.0037  0.0053  0.0105  0.0050

## 7. Leave-one-out modality utility per clip
For each clip: Δ_m = log p(y | T+A+V) − log p(y | all except m). Positive = modality m helped the correct label on that clip.
`share_helped` / `share_hurt` use a small tolerance (|Δ| > 0.01 nats) so that tiny numerical changes are not counted.

In [10]:
y_test = data["test"]["y"]; ALL = (0, 1, 2)
def logp_true(p): return np.log(np.clip(p[np.arange(N_TEST), y_test], 1e-12, 1.0))
delta = {}
rows = []
for (name, seed) in trained:
    if name.startswith("M1"):
        continue
    full = logp_true(subset_probs[(name, seed, ALL)])
    for i, m in enumerate(MODS):
        without = tuple(j for j in ALL if j != i)
        d = full - logp_true(subset_probs[(name, seed, without)])
        delta[(name, seed, m)] = d
        rows.append({"model": SHORT[name], "seed": seed, "modality": m, "mean_delta": d.mean(), "median_delta": np.median(d),
                     "share_helped": (d > 0.01).mean(), "share_hurt": (d < -0.01).mean(), "share_neutral": (np.abs(d) <= 0.01).mean()})
util = pd.DataFrame(rows); util.to_csv(f"{OUT_DIR}/loo_utility_summary.csv", index=False)
np.savez_compressed(f"{OUT_DIR}/loo_utility_per_clip.npz", **{f"{SHORT[n]}_seed{s}_{m}": v for (n, s, m), v in delta.items()})
util.groupby(["model", "modality"])[["mean_delta", "share_helped", "share_hurt", "share_neutral"]].mean().round(4)

mean_delta  share_helped  share_hurt  share_neutral
model modality                                                     
M2    audio        -0.0023        0.4461      0.4070         0.1470
      text          0.3299        0.7541      0.2388         0.0071
      vision        0.0046        0.4578      0.4054         0.1369
M3    audio        -0.0006        0.3489      0.3876         0.2636
      text          0.5079        0.7192      0.2692         0.0116
      vision        0.0153        0.4180      0.3873         0.1947
M4    audio        -0.0018        0.3494      0.3596         0.2910
      text          0.2922        0.7439      0.2483         0.0077
      vision        0.0086        0.3705      0.3537         0.2758

## 8. Do the gate weights follow the measured utility?
Spearman rank correlation, across test clips, between the gate weight a model gives modality m (all modalities present)
and Δ_m. Also the correlation between the gate weight and the model's own confidence (max probability).
A correlation near 0 means the gate does **not** track whether the modality actually helped on that clip.
This is a correlation, not a causal test.

In [11]:
rows = []
for (name, seed) in trained:
    if name.startswith("M1"):
        continue
    g = gates_full[(name, seed)]; conf = subset_probs[(name, seed, ALL)].max(1)
    for i, m in enumerate(MODS):
        rho_u, p_u = stats.spearmanr(g[:, i], delta[(name, seed, m)])
        rho_c, p_c = stats.spearmanr(g[:, i], conf)
        rows.append({"model": SHORT[name], "seed": seed, "modality": m, "mean_gate_weight": g[:, i].mean(),
                     "sd_gate_weight": g[:, i].std(), "spearman_gate_vs_utility": rho_u, "p_utility": p_u,
                     "spearman_gate_vs_confidence": rho_c, "p_confidence": p_c})
gate_util = pd.DataFrame(rows); gate_util.to_csv(f"{OUT_DIR}/gate_vs_utility.csv", index=False)
gate_util.round(4)

,model,seed,modality,mean_gate_weight,sd_gate_weight,spearman_gate_vs_utility,p_utility,spearman_gate_vs_confidence,p_confidence
0,M2,42,text,0.4348,0.0320,0.2137,0.0000,0.2758,0.0000
1,M2,42,audio,0.2814,0.0237,-0.0233,0.1110,-0.1379,0.0000
2,M2,42,vision,0.2838,0.0314,-0.0566,0.0001,-0.1735,0.0000
3,M2,43,text,0.4669,0.0132,0.1024,0.0000,0.0097,0.5099
4,M2,43,audio,0.2693,0.0148,0.0132,0.3681,-0.0103,0.4830
5,M2,43,vision,0.2638,0.0141,0.0802,0.0000,-0.0072,0.6244
6,M2,44,text,0.4308,0.0450,0.2674,0.0000,0.3201,0.0000
7,M2,44,audio,0.2860,0.0319,-0.0411,0.0050,-0.1769,0.0000
8,M2,44,vision,0.2832,0.0380,-0.0471,0.0013,-0.2183,0.0000
9,M3,42,text,0.6678,0.2170,0.1318,0.0000,0.1438,0.0000


## 9. Headroom: how often does dropping audio+video change the answer?
For every clip we compare the model's normal prediction (T+A+V) with its prediction from text alone (T).
- `fixed_by_dropping_AV`: wrong with T+A+V, correct with T only.
- `broken_by_dropping_AV`: correct with T+A+V, wrong with T only.
The **oracle** chooses, per clip, whichever of the two gives the correct label the higher probability. It uses the true label,
so it is only an **upper bound** for a rule that decides per clip whether to trust audio+video, using these trained models.
(An oracle over all 7 subsets is deliberately not reported: with 7 candidate predictions and only 3 classes it would find
a correct answer for most clips by chance, so it would say nothing about real headroom.)

In [12]:
rows = []
for (name, seed) in trained:
    if name.startswith("M1"):
        continue
    P_all, P_t = subset_probs[(name, seed, ALL)], subset_probs[(name, seed, (0,))]
    ok_all, ok_t = P_all.argmax(1) == y_test, P_t.argmax(1) == y_test
    use_t = P_t[np.arange(N_TEST), y_test] > P_all[np.arange(N_TEST), y_test]
    oracle_p = np.where(use_t[:, None], P_t, P_all)
    rows.append({"model": SHORT[name], "seed": seed,
                 "macro_f1_TAV": evaluate(P_all, data["test"])["macro_f1"], "macro_f1_T_only": evaluate(P_t, data["test"])["macro_f1"],
                 "macro_f1_oracle_TAV_or_T": evaluate(oracle_p, data["test"])["macro_f1"],
                 "fixed_by_dropping_AV": int((~ok_all & ok_t).sum()), "broken_by_dropping_AV": int((ok_all & ~ok_t).sum()),
                 "prediction_changes": int((P_all.argmax(1) != P_t.argmax(1)).sum())})
oracle = pd.DataFrame(rows); oracle.to_csv(f"{OUT_DIR}/av_headroom.csv", index=False)
print(f"Test clips: {N_TEST}")
oracle.round(4)

Test clips: 4659


,model,seed,macro_f1_TAV,macro_f1_T_only,macro_f1_oracle_TAV_or_T,fixed_by_dropping_AV,broken_by_dropping_AV,prediction_changes
0,M2,42,0.6272,0.6190,0.6727,230,245,562
1,M2,43,0.6169,0.6164,0.6503,166,167,396
2,M2,44,0.6240,0.6142,0.6645,199,232,520
3,M3,42,0.6276,0.6225,0.6796,235,282,595
4,M3,43,0.6397,0.6280,0.6723,136,242,436
5,M3,44,0.6188,0.6139,0.6493,150,156,354
6,M4,42,0.6373,0.6256,0.6486,54,116,208
7,M4,43,0.6287,0.6236,0.6531,102,163,309
8,M4,44,0.6374,0.6309,0.6663,123,198,363


## 10. Repeatability diagnostic (optional, short re-training)
v2 reproduced M2 exactly but not M1, M3, M4 (largest macro-F1 difference 0.0057). Code differences are ruled out:
the v1 and v2 code gave bit-identical results when tested on the same machine.
**Hypothesis to test (not yet established):** the number of CPU threads (or the machine/library version) changes the order
of floating-point additions, which changes the trained weights slightly.

Test: re-train M1 seed 42, M2 seed 42 and M3 seed 44 (the fastest runs) with 1, 2 and the default number of threads, here,
and compare with the v2 checkpoint (same machine type) and the v1 result. Weights are compared by a hash of their bytes.
The v1 machine's thread count was not recorded, so this test can **support or weaken** the hypothesis, not prove the v1 cause.

In [13]:
if RUN_REPEATABILITY:
    default_threads = torch.get_num_threads()
    thread_list = sorted({1, 2, default_threads})
    v1 = pd.read_csv(V1_RESULTS) if os.path.exists(V1_RESULTS) else None
    def wh(state): return hashlib.md5(b"".join(v.numpy().tobytes() for v in state.values())).hexdigest()[:12]
    rows = []
    for name, seed in [("M1 Text-only", 42), ("M2 EMO-FAR RMSR (as written)", 42), ("M3 Fixed fusion", SEEDS[-1])]:
        v2_state = torch.load(f"{CKPT_DIR}/{SHORT[name]}_seed{seed}.pt", map_location="cpu")
        for nt in thread_list:
            torch.set_num_threads(nt); t1 = time.time()
            model, best, ep = train(name, seed)
            f1 = evaluate(predict(model, data["test"]), data["test"])["macro_f1"]
            row = {"model": SHORT[name], "seed": seed, "threads": nt, "test_macro_f1": f1, "stopped_epoch": ep,
                   "weights_hash": wh(model.state_dict()), "v2_weights_hash": wh(v2_state),
                   "same_weights_as_v2": wh(model.state_dict()) == wh(v2_state), "seconds": round(time.time() - t1, 1)}
            if v1 is not None:
                row["v1_test_macro_f1"] = float(v1[(v1.model == name) & (v1.seed == seed)].macro_f1.iloc[0])
                row["equals_v1"] = abs(row["v1_test_macro_f1"] - f1) < 1e-9
            rows.append(row)
    torch.set_num_threads(default_threads)
    rep = pd.DataFrame(rows); rep.to_csv(f"{OUT_DIR}/repeatability_threads.csv", index=False)
    print(rep.to_string(index=False))
else:
    print("Skipped (RUN_REPEATABILITY = False)")

  M1 Text-only                           seed 42: best valid macro-F1 = 0.6246  (stopped after epoch 12)
  M1 Text-only                           seed 42: best valid macro-F1 = 0.6244  (stopped after epoch 12)
  M1 Text-only                           seed 42: best valid macro-F1 = 0.6244  (stopped after epoch 12)
  M2 EMO-FAR RMSR (as written)           seed 42: best valid macro-F1 = 0.6181  (stopped after epoch 9)
  M2 EMO-FAR RMSR (as written)           seed 42: best valid macro-F1 = 0.6181  (stopped after epoch 9)
  M2 EMO-FAR RMSR (as written)           seed 42: best valid macro-F1 = 0.6181  (stopped after epoch 9)
  M3 Fixed fusion                        seed 44: best valid macro-F1 = 0.6256  (stopped after epoch 7)
  M3 Fixed fusion                        seed 44: best valid macro-F1 = 0.6256  (stopped after epoch 7)
  M3 Fixed fusion                        seed 44: best valid macro-F1 = 0.6256  (stopped after epoch 7)
model  seed  threads  test_macro_f1  stopped_epoch weights_ha

## 11. Files written

In [14]:
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn, f"{os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e3:.0f} kB")

   av_headroom.csv 1 kB
   gate_vs_utility.csv 3 kB
   integrity_check.csv 0 kB
   loo_utility_per_clip.npz 465 kB
   loo_utility_summary.csv 3 kB
   repeatability_threads.csv 1 kB
   run_info.json 1 kB
   subset_evaluation.csv 7 kB
   test_ids.csv 69 kB
   test_labels.npy 37 kB
   test_scores.npy 19 kB
   valid_labels.npy 15 kB
